# Solar Filament — Fixed Inference
Loads best.pt, uses v12 thresholds, writes submission.csv

In [ ]:
import os, glob, shutil, sys

# 1. Locate best.pt from any attached dataset
ckpt_candidates = sorted(glob.glob('/kaggle/input/**/best.pt', recursive=True))
print('best.pt candidates:', ckpt_candidates)
assert ckpt_candidates, 'best.pt not found — attach the dataset with weights'
shutil.copy(ckpt_candidates[0], '/kaggle/working/best.pt')
print('copied to /kaggle/working/best.pt')

# 2. Clone repo
os.system('pip install -q segmentation-models-pytorch pycocotools')
os.system('git clone --depth 1 https://github.com/maurolima-sys/solar-filament-segmentation.git /kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')

In [ ]:
import filament as F
import torch, numpy as np

print('device:', F.dev)
print('ckpt exists:', os.path.exists(F.CFG['ckpt']))

# 3. Override ROOT path (competition data is at /kaggle/input/filament-segmentation-2026/)
roots = glob.glob('/kaggle/input/**/MAGFiLO_1.0_Kaggle_2026', recursive=True)
print('data roots found:', roots)
assert roots, 'Competition data not found'
ROOT = roots[0]
F.CFG.update(
    ann_json=f'{ROOT}/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json',
    train_dir=f'{ROOT}/train/train_images',
    test_dir=f'{ROOT}/test/test_images',
)
print('test images:', len(glob.glob(F.CFG['test_dir'] + '/*')))

In [ ]:
# 4. Load model
model = F.build(pretrained=False).to(F.dev)
model.load_state_dict(torch.load(F.CFG['ckpt'], map_location=F.dev, weights_only=True))
model.eval()
print('model loaded')

In [ ]:
# 5. Probability diagnostics on 3 test images
test_paths = sorted(glob.glob(F.CFG['test_dir'] + '/*'))[:3]
for p in test_paths:
    x = F.TestDS([p])[0][None].to(F.dev)
    pr = F.tta_predict(model, x)
    print(f'{os.path.basename(p)}: min={pr.min():.4f} mean={pr.mean():.4f} '
          f'p95={np.percentile(pr,95):.4f} max={pr.max():.4f} '
          f'px>0.3={int((pr>0.3).sum())} px>0.5={int((pr>0.5).sum())}')

In [ ]:
# 6. Set v12 thresholds and predict
F.CFG.update(t_lo=0.35, t_hi=0.50, min_meanp=0.30, min_area=50)
print('thresholds:', {k: F.CFG[k] for k in ('t_lo','t_hi','min_meanp','min_area')})
F.predict('/kaggle/working/submission.csv')

In [ ]:
# 7. Validate
import pandas as pd
from pycocotools import mask as mu

df = pd.read_csv('/kaggle/working/submission.csv')
print(f'{len(df)} rows, {df["filament_id"].str.rsplit("_",n=1).str[0].nunique()} images')
print('columns ok:', list(df.columns) == ['filament_id', 'segmentation_rle'])

# decode check on first 5
for _, row in df.head(5).iterrows():
    rle_obj = {'size': [2048, 2048], 'counts': row['segmentation_rle'].encode()}
    m = mu.decode(rle_obj)
    print(f'  {row["filament_id"]}: {int(m.sum())} pixels')